# 09 · Factorial architecture × tokenizer mechanism evidence

v0.7 moves from asking whether one discovered circuit survives held-out validation to asking whether **architecture or tokenization changes the held-out mechanism under a matched design**.

The rule for this notebook is simple: **estimability comes before effect size**. A large difference is not a tokenizer effect if token budget, compute, task performance, checkpoint maturity, split semantics, or evidence protocol also changed.

## Hypothesis

We will use the package's known-ground-truth factorial benchmark. It contains two architectures, two tokenizers, and two neural-session replicas. One architecture behaves similarly under both tokenizers, while the second architecture loses 0.5 held-out joint faithfulness under the second tokenizer.

Before looking at results, the prediction is therefore:

```text
architecture × tokenizer interaction = -0.5
```

in both sessions. The same gate also contains a token-budget-confounded tokenizer comparison and a 2×2 grid with one missing cell. Those should be refused, not estimated.

In [ ]:
from neuros_mechint.benchmarks import run_factorial_ground_truth_benchmark

gate = run_factorial_ground_truth_benchmark()
gate.to_dict()

A correct implementation should report two observed interactions near `-0.5`, cross-session replication readiness, and successful rejection of both bad designs. Notice that rejecting a comparison is a **successful scientific result** when the design cannot support the requested estimate.

## Build an explicit 2×2 design

The next cell shows the design objects directly. In a real study, each observed cell would be backed by a completed v0.6 `EvidencePackResult`. Semantic partition IDs are part of the cell because different tokenizers can encode the same trials into different raw tensors.

In [ ]:
from neuros_mechint.benchmarks import (
    FactorialAnalysisPolicy, FactorialCellOutcome, FactorialCellSpec,
    FactorialMechanismSpec, MatchedCovariate, analyze_factorial_mechanisms,
    preregister_2x2_contrasts,
)

def cell(architecture, tokenizer):
    return FactorialCellSpec(
        cell_id=f'{architecture}:{tokenizer}', architecture=architecture,
        tokenizer_id=tokenizer, model_id=architecture, model_revision=f'{architecture}-rev',
        tokenizer_revision=f'{tokenizer}-rev', dataset_id='demo', dataset_revision='demo-v1',
        session_id='session-1', subject_id='animal-1', metric_name='score',
        discovery_method='fixed-candidate', discovery_partition_id='session-1:discovery',
        validation_partition_id='session-1:validation', training_seed=0, checkpoint='step:100',
        checkpoint_maturity=1.0, target_universe=('route_a', 'route_b'),
        covariates={'token_budget': 128, 'temporal_resolution_ms': 10.0,
                    'downstream_capacity': 32, 'training_compute': 1000},
    )

cells = tuple(cell(a, t) for a in ('transformer', 'ssm') for t in ('event', 'relative-isi'))
contrasts = preregister_2x2_contrasts(
    prefix='session-1', architectures=('transformer', 'ssm'),
    tokenizers=('event', 'relative-isi'),
    fixed_axes={'dataset_id': 'demo', 'session_id': 'session-1', 'subject_id': 'animal-1',
                'training_seed': 0, 'checkpoint': 'step:100'},
)
spec = FactorialMechanismSpec(
    study_id='demo-factorial', cells=cells, contrasts=contrasts,
    matched_covariates=(MatchedCovariate('token_budget'), MatchedCovariate('temporal_resolution_ms'),
                        MatchedCovariate('downstream_capacity'), MatchedCovariate('training_compute')),
    policy=FactorialAnalysisPolicy(max_task_metric_delta=0.02),
)
len(spec.cells), len(spec.contrasts)

`preregister_2x2_contrasts` creates five direct questions: architecture at tokenizer 1, architecture at tokenizer 2, tokenizer at architecture 1, tokenizer at architecture 2, and the architecture×tokenizer difference-in-differences interaction. There is intentionally no omnibus mechanism score.

In [ ]:
def outcome(joint, task=0.80):
    return FactorialCellOutcome(
        task_metric=task, candidate_size=1, validation_sufficiency=joint,
        validation_necessity=joint, validation_joint_faithfulness=joint,
        validation_joint_random_percentile=1.0, discovery_to_validation_drop=0.02,
        intervention_baseline_sensitivity=0.01, promotion_passed=joint >= 0.5,
        source_study_fingerprint=f'study-{joint}', source_run_hash=f'run-{joint}',
        evidence_protocol_fingerprint='same-v06-protocol',
        effect_map={'route_a': joint, 'route_b': 1-joint},
    )

outcomes = {
    'transformer:event': outcome(0.90),
    'transformer:relative-isi': outcome(0.90, 0.81),
    'ssm:event': outcome(0.90),
    'ssm:relative-isi': outcome(0.40, 0.81),
}
report = analyze_factorial_mechanisms(spec, outcomes)
[(c.contrast_id, c.estimable, c.outcome_effects.get('validation_joint_faithfulness'))
 for c in report.contrasts]

## Falsification: break token-budget matching

A tokenizer comparison becomes scientifically ambiguous when the tokenizer also changes the amount of information or compute exposed to the downstream model. In a real experiment, change one cell's `token_budget` from 128 to 64 and rerun the same contrast. v0.7 should return `estimable=False` with a matched-covariate reason instead of reporting a tokenizer effect.

That refusal is one of the main features of the release.

## What to do with real neural data

Start small: two architectures × two tokenizers × two sessions. Use the same semantic discovery/validation trial partitions across cells, match token budget, temporal resolution, model capacity, training compute, task performance, and checkpoint maturity, and generate a v0.6 evidence pack for each cell. Only then run the factorial contrast layer.

A good first comparison is `Transformer × SSM` crossed with `event × relative-ISI`. If an interaction survives held-out evidence and cross-session replication, v0.8 can ask whether the conditions learned genuinely different causal features or merely different representational bases.